# 🧠 SALIENT: fNIRS Mental Workload Classification

**Author:** HM Jemima  
**Dataset:** Tufts fNIRS2MW  
**Version:** Verbose Output (shows training progress)

---

In [ ]:
#@title 1. Setup & Imports

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, cohen_kappa_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
import warnings
import glob
import json
import pickle
import joblib
from datetime import datetime
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("="*60)
print("SALIENT - Setup Complete")
print("="*60)
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print(f"Time: {datetime.now().strftime('%H:%M:%S')}")
print("="*60)

In [ ]:
#@title 2. Configuration

WINDOW_SIZE = '30sec_150ts'

# ========== KAGGLE PATHS ==========
BASE_PATH = '/kaggle/input/datasets/hmjemima/tufts-fnirs2mw/slide_window_data/'
OUTPUT_PATH = '/kaggle/working/SALIENT_Results/'

CONFIG = {
    'data_path': f'{BASE_PATH}size_{WINDOW_SIZE}_stride_03ts/',
    'output_path': OUTPUT_PATH,
    'window_size_key': WINDOW_SIZE,
    
    'quick_test': False,
    'max_subjects': None,
    
    'n_channels': 8,
    'n_classes': 4,
    'timesteps': 150,
    
    'feature_cols': ['AB_I_O', 'AB_PHI_O', 'AB_I_DO', 'AB_PHI_DO', 
                     'CD_I_O', 'CD_PHI_O', 'CD_I_DO', 'CD_PHI_DO'],
    
    # Training
    'batch_size': 32,
    'max_epochs': 100,
    'patience': 20,
    'learning_rate': 0.0005,
    
    # Model
    'embed_dim': 64,
    'n_heads': 4,
    'dropout': 0.4,
}

# Create output dirs
for d in ['', 'figures']:
    os.makedirs(os.path.join(OUTPUT_PATH, d), exist_ok=True)

print("✅ Config loaded")
print(f"📂 Data: {CONFIG['data_path']}")

In [ ]:
#@title 3. Verify Data Path

if os.path.exists(CONFIG['data_path']):
    csv_files = glob.glob(os.path.join(CONFIG['data_path'], "*.csv"))
    print(f"✅ Found {len(csv_files)} CSV files")
else:
    print("❌ Path not found. Searching...")
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.csv') for f in files):
            print(f"   Found CSVs in: {root}")

In [ ]:
#@title 4. Load Dataset

def load_dataset(config):
    all_X, all_y, all_subjects = [], [], []
    
    csv_files = sorted(glob.glob(os.path.join(config['data_path'], "*.csv")))
    
    if config.get('max_subjects'):
        csv_files = csv_files[:config['max_subjects']]
    
    print(f"Loading {len(csv_files)} subjects...")
    
    for idx, filepath in enumerate(tqdm(csv_files)):
        subject_id = idx + 1
        df = pd.read_csv(filepath)
        
        for chunk_id in df['chunk'].unique():
            chunk = df[df['chunk'] == chunk_id]
            X = chunk[config['feature_cols']].values.astype(np.float32)
            label = int(chunk['label'].iloc[0])
            
            if len(X) >= config['timesteps']:
                X = X[:config['timesteps']]
            else:
                X = np.pad(X, ((0, config['timesteps'] - len(X)), (0, 0)), mode='edge')
            
            all_X.append(X)
            all_y.append(label)
            all_subjects.append(subject_id)
    
    X = np.array(all_X, dtype=np.float32)
    y = np.array(all_y, dtype=np.int32)
    subjects = np.array(all_subjects, dtype=np.int32)
    
    X = np.nan_to_num(X)
    
    return X, y, subjects

X, y, subjects = load_dataset(CONFIG)

print(f"\n✅ Data loaded!")
print(f"   X shape: {X.shape}")
print(f"   Subjects: {len(np.unique(subjects))}")
print(f"   Samples: {len(y)}")
print(f"   Classes: {np.bincount(y)}")

In [ ]:
#@title 5. Custom Layers

class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.embed_dim = embed_dim
    
    def build(self, input_shape):
        self.pos_emb = self.add_weight(
            shape=(self.max_len, self.embed_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='pos_emb'
        )
    
    def call(self, x):
        return x + self.pos_emb[:tf.shape(x)[1], :]
    
    def get_config(self):
        return {'max_len': self.max_len, 'embed_dim': self.embed_dim}


class ChannelAttention(layers.Layer):
    def __init__(self, reduction=4, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
    
    def build(self, input_shape):
        ch = input_shape[-1]
        self.fc1 = layers.Dense(max(ch // self.reduction, 1), activation='relu')
        self.fc2 = layers.Dense(ch, activation='sigmoid')
    
    def call(self, x):
        avg = tf.reduce_mean(x, axis=1, keepdims=True)
        attn = self.fc2(self.fc1(avg))
        return x * attn

    def get_config(self):
        config = super().get_config()
        config.update({'reduction': self.reduction})
        return config


class AttentionPooling(layers.Layer):
    def build(self, input_shape):
        self.dense = layers.Dense(1, activation='tanh')
    
    def call(self, x):
        w = tf.nn.softmax(self.dense(x), axis=1)
        return tf.reduce_sum(x * w, axis=1)

    def get_config(self):
        return super().get_config()


print("✅ Custom layers defined")

In [ ]:
#@title 6. SALIENT Model

def build_salient(input_shape, n_classes, config):
    inputs = layers.Input(shape=input_shape)
    
    # Embedding
    x = layers.Conv1D(config['embed_dim'], 1, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = PositionalEncoding(input_shape[0], config['embed_dim'])(x)
    
    # Spatial stream
    spatial = ChannelAttention()(x)
    spatial = layers.Conv1D(config['embed_dim'], 3, padding='same', activation='relu')(spatial)
    spatial = layers.BatchNormalization()(spatial)
    
    # Temporal stream
    temporal = layers.MultiHeadAttention(
        num_heads=config['n_heads'],
        key_dim=config['embed_dim'] // config['n_heads']
    )(x, x)
    temporal = layers.LayerNormalization()(x + temporal)
    temporal = layers.Conv1D(config['embed_dim'], 3, padding='same', activation='relu')(temporal)
    temporal = layers.BatchNormalization()(temporal)
    
    # Fusion
    fused = layers.Concatenate()([spatial, temporal])
    fused = layers.Conv1D(config['embed_dim'], 1, activation='relu')(fused)
    fused = layers.Dropout(config['dropout'])(fused)
    
    # BiLSTM
    lstm = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(fused)
    lstm = layers.Dropout(config['dropout'])(lstm)
    
    # Pooling + Classification
    context = AttentionPooling()(lstm)
    x = layers.Dense(64, activation='relu')(context)
    x = layers.Dropout(config['dropout'])(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    
    return Model(inputs, outputs, name='SALIENT')

# Test
test_model = build_salient((CONFIG['timesteps'], CONFIG['n_channels']), CONFIG['n_classes'], CONFIG)
print(f"✅ SALIENT: {test_model.count_params():,} parameters")
del test_model
K.clear_session()

In [ ]:
#@title 7. Progress Callback (Shows Training Progress)

class ProgressCallback(Callback):
    """Custom callback to show training progress every 10 epochs."""
    
    def __init__(self, fold_num, total_folds):
        super().__init__()
        self.fold_num = fold_num
        self.total_folds = total_folds
        self.start_time = None
    
    def on_train_begin(self, logs=None):
        self.start_time = datetime.now()
        print(f"      Training started at {self.start_time.strftime('%H:%M:%S')}")
    
    def on_epoch_end(self, epoch, logs=None):
        # Print every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0:
            loss = logs.get('loss', 0)
            acc = logs.get('accuracy', 0) * 100
            val_loss = logs.get('val_loss', 0)
            val_acc = logs.get('val_accuracy', 0) * 100
            print(f"      Epoch {epoch+1:3d}: loss={loss:.4f}, acc={acc:.1f}%, val_loss={val_loss:.4f}, val_acc={val_acc:.1f}%")
    
    def on_train_end(self, logs=None):
        duration = datetime.now() - self.start_time
        print(f"      Training completed in {duration.total_seconds():.1f}s")

print("✅ Progress callback defined")

In [ ]:
#@title 8. LOSO Training (WITH VERBOSE OUTPUT)

def train_loso(X, y, subjects, model_fn, config):
    """LOSO with verbose progress output."""
    
    logo = LeaveOneGroupOut()
    unique_subj = np.unique(subjects)
    n_subjects = len(unique_subj)
    
    all_y_true = []
    all_y_pred = []
    subject_results = []
    
    print("\n" + "="*70)
    print(f"LOSO CROSS-VALIDATION | {n_subjects} subjects")
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    
    fold = 0
    for train_idx, test_idx in logo.split(X, y, subjects):
        fold += 1
        test_subj = subjects[test_idx[0]]
        
        print(f"\n{'─'*70}")
        print(f"📊 FOLD {fold}/{n_subjects} | Test Subject: {test_subj}")
        print(f"   Train samples: {len(train_idx)} | Test samples: {len(test_idx)}")
        print(f"{'─'*70}")
        
        # Split
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Normalize
        scaler = StandardScaler()
        X_train_flat = X_train.reshape(len(X_train), -1)
        X_test_flat = X_test.reshape(len(X_test), -1)
        X_train_norm = scaler.fit_transform(X_train_flat).reshape(X_train.shape)
        X_test_norm = scaler.transform(X_test_flat).reshape(X_test.shape)
        
        # Build model
        model = model_fn((X_train.shape[1], X_train.shape[2]), config['n_classes'], config)
        model.compile(
            optimizer=keras.optimizers.Adam(config['learning_rate']),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Callbacks
        callbacks = [
            ProgressCallback(fold, n_subjects),
            EarlyStopping(
                monitor='val_loss',
                patience=config['patience'],
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=8,
                min_lr=1e-6,
                verbose=1
            )
        ]
        
        # Train with verbose=0 (our callback handles output)
        model.fit(
            X_train_norm, y_train,
            validation_split=0.15,
            batch_size=config['batch_size'],
            epochs=config['max_epochs'],
            callbacks=callbacks,
            verbose=0
        )
        
        # Evaluate
        y_pred = np.argmax(model.predict(X_test_norm, verbose=0), axis=1)
        
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        print(f"\n   ✅ FOLD {fold} RESULT: Accuracy = {acc*100:.2f}% | F1 = {f1*100:.2f}%")
        
        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)
        subject_results.append({'subject': int(test_subj), 'accuracy': acc, 'f1': f1})
        
        # Clear memory
        K.clear_session()
        del model
        
        # Show running average
        running_acc = np.mean([r['accuracy'] for r in subject_results]) * 100
        print(f"   📈 Running Average Accuracy: {running_acc:.2f}% ({fold}/{n_subjects} folds)")
    
    # Final results
    all_y_true = np.array(all_y_true)
    all_y_pred = np.array(all_y_pred)
    
    overall_acc = accuracy_score(all_y_true, all_y_pred)
    overall_f1 = f1_score(all_y_true, all_y_pred, average='weighted')
    overall_kappa = cohen_kappa_score(all_y_true, all_y_pred)
    
    subj_accs = [r['accuracy'] for r in subject_results]
    mean_acc = np.mean(subj_accs)
    std_acc = np.std(subj_accs)
    ci_95 = 1.96 * std_acc / np.sqrt(len(subj_accs))
    
    print("\n" + "="*70)
    print("🎉 FINAL RESULTS")
    print("="*70)
    print(f"   Overall Accuracy:  {overall_acc*100:.2f}%")
    print(f"   Overall F1 Score:  {overall_f1*100:.2f}%")
    print(f"   Cohen's Kappa:     {overall_kappa:.4f}")
    print(f"   Mean ± Std:        {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")
    print(f"   95% CI:            [{(mean_acc-ci_95)*100:.2f}%, {(mean_acc+ci_95)*100:.2f}%]")
    print("="*70)
    
    return {
        'overall_accuracy': overall_acc,
        'overall_f1': overall_f1,
        'overall_kappa': overall_kappa,
        'mean_accuracy': mean_acc,
        'std_accuracy': std_acc,
        'ci_95': ci_95,
        'subject_results': subject_results,
        'confusion_matrix': confusion_matrix(all_y_true, all_y_pred).tolist(),
        'y_true': all_y_true.tolist(),
        'y_pred': all_y_pred.tolist()
    }

print("✅ LOSO training function defined")

In [ ]:
#@title 9. 🚀 RUN TRAINING (SHOWS PROGRESS)

print("\n" + "#"*70)
print("# STARTING SALIENT TRAINING")
print("# You will see progress for each epoch and each fold!")
print("#"*70)

results = train_loso(X, y, subjects, build_salient, CONFIG)

In [ ]:
#@title 10. Save Results

# Save pickle
pkl_path = os.path.join(OUTPUT_PATH, f'results_{WINDOW_SIZE}.pkl')
save_data = {
    'experiment': 'SALIENT',
    'author': 'HM Jemima',
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'config': CONFIG,
    'results': results
}
with open(pkl_path, 'wb') as f:
    pickle.dump(save_data, f)
print(f"✅ Saved: {pkl_path}")

# Save JSON
json_path = os.path.join(OUTPUT_PATH, f'summary_{WINDOW_SIZE}.json')
summary = {
    'accuracy': f"{results['overall_accuracy']*100:.2f}%",
    'f1': f"{results['overall_f1']*100:.2f}%",
    'kappa': f"{results['overall_kappa']:.4f}",
    'mean_std': f"{results['mean_accuracy']*100:.2f}% ± {results['std_accuracy']*100:.2f}%"
}
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ Saved: {json_path}")

# --- Train final deployment model on ALL data ---
print("\n🚀 Training final deployment model on ALL data...")
sc_final = StandardScaler()
X_scaled = sc_final.fit_transform(X.reshape(len(X), -1)).reshape(X.shape)

final_model = build_salient((X.shape[1], X.shape[2]), CONFIG['n_classes'], CONFIG)
final_model.compile(optimizer=keras.optimizers.Adam(CONFIG['learning_rate']),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
final_model.fit(X_scaled, y, validation_split=0.1,
                batch_size=CONFIG['batch_size'], epochs=CONFIG['max_epochs'], verbose=1,
                callbacks=[EarlyStopping('val_loss', patience=CONFIG['patience'], restore_best_weights=True),
                           ReduceLROnPlateau('val_loss', factor=0.5, patience=8, min_lr=1e-6)])

final_model.save(os.path.join(OUTPUT_PATH, 'SALIENT_model.h5'))
joblib.dump(sc_final, os.path.join(OUTPUT_PATH, 'SALIENT_scaler.pkl'))

metadata = {
    'model_type': 'SALIENT',
    'version': 'v1.0.0',
    'accuracy': f'{results["overall_accuracy"]*100:.2f}%',
    'f1_score': f'{results["overall_f1"]*100:.2f}%',
    'kappa': round(results['overall_kappa'], 4),
    'features': '1,200 (8 fNIRS channels x 150 timesteps)',
    'training_data': f'Tufts fNIRS2MW — {len(np.unique(subjects))} subjects, LOSO CV',
    'n_classes': CONFIG['n_classes'],
    'class_labels': ['0-back', '1-back', '2-back', '3-back'],
    'input_shape': [CONFIG['timesteps'], len(CONFIG['feature_cols'])],
    'last_updated': datetime.now().strftime('%Y-%m-%d')
}
with open(os.path.join(OUTPUT_PATH, 'SALIENT_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Exported: model (.h5), scaler (.pkl), metadata (.json)")

In [ ]:
#@title 11. Visualizations

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
ax1 = axes[0]
cm = np.array(results['confusion_matrix'])
cm_norm = cm / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues', ax=ax1,
            xticklabels=['0-back', '1-back', '2-back', '3-back'],
            yticklabels=['0-back', '1-back', '2-back', '3-back'])
ax1.set_title(f'SALIENT Confusion Matrix\nAccuracy: {results["overall_accuracy"]*100:.2f}%')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')

# Per-subject accuracy
ax2 = axes[1]
subj_accs = [r['accuracy']*100 for r in results['subject_results']]
ax2.bar(range(1, len(subj_accs)+1), subj_accs, color='steelblue', alpha=0.7)
ax2.axhline(72.2, color='green', linestyle='--', linewidth=2, label='Tufts Benchmark (72.2%)')
ax2.axhline(results['overall_accuracy']*100, color='red', linestyle='-', linewidth=2, label=f'SALIENT ({results["overall_accuracy"]*100:.1f}%)')
ax2.set_xlabel('Subject')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Per-Subject Accuracy')
ax2.legend()
ax2.set_ylim(0, 100)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_PATH, 'figures', 'results.png')
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"✅ Figure saved: {fig_path}")

In [ ]:
#@title 12. Final Summary

print("\n" + "="*70)
print("🎉 EXPERIMENT COMPLETE")
print("="*70)
print(f"\n📊 SALIENT Results:")
print(f"   Accuracy:     {results['overall_accuracy']*100:.2f}%")
print(f"   F1 Score:     {results['overall_f1']*100:.2f}%")
print(f"   Kappa:        {results['overall_kappa']:.4f}")
print(f"   95% CI:       [{(results['mean_accuracy']-results['ci_95'])*100:.2f}%, {(results['mean_accuracy']+results['ci_95'])*100:.2f}%]")

print(f"\n📊 Comparison with Tufts Benchmark:")
print(f"   Tufts RF:     72.20%")
print(f"   SALIENT:      {results['overall_accuracy']*100:.2f}%")
print(f"   Improvement:  {(results['overall_accuracy']*100 - 72.2):+.2f}%")

print(f"\n📁 Output files:")
for f in os.listdir(OUTPUT_PATH):
    print(f"   • {f}")

print("\n" + "="*70)
print(f"Author: HM Jemima | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

In [ ]:
#@title 13. Download Results (Create ZIP)

import shutil
shutil.make_archive('/kaggle/working/SALIENT_Results', 'zip', OUTPUT_PATH)
print("✅ Created: /kaggle/working/SALIENT_Results.zip")
print("\n📥 Click on the file in Output panel (right side) to download!")